# CommNet

> CommNet (Sukhbaatar et al., 2016): agents that communicate by averaging hidden states, trained end to end online (REINFORCE) or from datasets.

In [ ]:
#| default_exp wm.commnet.core

In [ ]:
#| hide
from nbdev.showdoc import *

[CommNet](https://arxiv.org/abs/1605.07736) ("Learning Multiagent Communication with
Backpropagation", Sukhbaatar, Szlam and Fergus, NeurIPS 2016) controls all the agents with one
network whose layers exchange continuous messages. Each agent encodes its observation,
`h_i^0 = r(s_i)`, then `K` communication steps update every agent's hidden state:

* `h_i^{j+1} = tanh(H^j h_i^j + C^j c_i^j + S^j h_i^0)` (the last term: the paper's skip connection);
* `c_i^{j+1}` is the mean of the other agents' `h^{j+1}` (the agents agent `i` hears), and `c_i^0 = 0`.

so agents exchange messages `K - 1` times (`comm_steps=1`: independent agents, the paper's
"no communication" baseline). Each agent's action is sampled from a softmax over `h_i^K`. The
weights are shared by the agents, and messages are differentiable, so the whole team is trained
end to end: online with REINFORCE and a learned baseline as in the paper (`train_commnet`), or
from a dataset with the expert's actions as labels (`train_commnet_supervised`).

The agents an agent hears are the alive ones (agents that reached their goal leave the
communication, as agents leaving the paper's traffic junction), optionally only those within
`comm_range` cells (Chebyshev distance, e.g. 3: inside a 7 x 7 view), and with probability
`drop_prob` a message is lost (each sender-receiver pair, each step). The messages can also be
sent over comm-core networks (`NetworkLinks`: `stable_marl.comm`'s `AgentNetwork`, one per env), whose
topology, channel and protocol decide which messages arrive, round by round. The same masks are used
when acting and when training.

As in the paper's grid tasks (a lookup table of each agent's local view), the observations are
features: each agent's `obs_keys` (default: its symbolic 7 x 7 view `image` and `direction`, as
the MAMBA baseline) and, with `goal`, its offset to its goal cell.

In [ ]:
#| export
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch

from stable_marl.comm import AgentNetwork, within_range
from stable_marl.world.env_pool import episode_done
from stable_marl.wm.omnivla.core import IGNORE_INDEX, shortest_path_actions, walkable_cells

### Observations

In [ ]:
#| export
_IMAGE_SCALE = 10.0   # symbolic cells (type, color, state) scaled to about [0, 1]


def commnet_features(
    columns: dict[str, np.ndarray], # Columns of one step, (B, num_agents, ...) (e.g. an info dict's last step)
    obs_keys: Sequence[str] = ('image', 'direction'), # Each agent's observations
    grid_size: int = 15, # Side of the grid (scales positions)
    goal: bool = True, # Add each agent's offset to its goal cell (`goal_position - position`)
) -> np.ndarray:
    "Each agent's observation features (B, num_agents, dim), float32."
    parts = []
    for key in obs_keys:
        value = np.asarray(columns[key])
        lead = value.shape[:2]
        if key == 'direction':
            parts.append(np.eye(4, dtype=np.float32)[value.astype(np.int64)])
        elif key == 'position':
            parts.append(value.astype(np.float32) / grid_size)
        elif key == 'image':
            parts.append(value.reshape(*lead, -1).astype(np.float32) / _IMAGE_SCALE)
        elif key == 'pixels':
            parts.append(value.reshape(*lead, -1).astype(np.float32) / 255.0)
        else:
            parts.append(value.reshape(*lead, -1).astype(np.float32))
    if goal:
        offset = np.asarray(columns['goal_position'], np.float32) - np.asarray(columns['position'], np.float32)
        parts.append(offset / grid_size)
    return np.concatenate(parts, -1).astype(np.float32)

### Who hears whom

In [ ]:
#| export
def comm_mask(
    alive: torch.Tensor, # (B, N) bool: agents still acting
    position: torch.Tensor | None = None, # (B, N, 2) cells, needed with `comm_range`
    comm_range: int | None = None, # Agents hear the agents within this Chebyshev distance (None: all)
    drop_prob: float = 0.0, # Probability that a message is lost (each sender-receiver pair)
) -> torch.Tensor:
    "(B, N, N) bool: `mask[b, i, k]` when agent `i` hears agent `k` (`k != i` is the caller's concern)."
    alive = torch.as_tensor(alive, dtype=torch.bool)
    mask = alive.unsqueeze(1) & alive.unsqueeze(2)
    if comm_range is not None:
        position = torch.as_tensor(position)
        distance = (position.unsqueeze(2) - position.unsqueeze(1)).abs().amax(-1)
        mask = mask & (distance <= comm_range)
    if drop_prob > 0:
        mask = mask & (torch.rand(mask.shape, device=mask.device) >= drop_prob)
    return mask


def mean_of_others(h: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
    "`c_i`: the mean of `h` (B, N, D) over the other agents agent `i` hears (`mask` (B, N, N)), 0 if none."
    B, N, _ = h.shape
    hears = torch.ones(B, N, N, dtype=torch.bool, device=h.device) if mask is None else mask.bool()
    hears = hears & ~torch.eye(N, dtype=torch.bool, device=h.device)
    weights = hears.to(h.dtype)
    return weights @ h / weights.sum(-1, keepdim=True).clamp(min=1)

In [ ]:
#| hide
from fastcore.test import test_eq, test_close
h = torch.arange(12, dtype=torch.float32).view(1, 3, 4)
test_close(mean_of_others(h)[0, 0], (h[0, 1] + h[0, 2]) / 2)
alive = torch.tensor([[True, True, False]])
test_close(mean_of_others(h, comm_mask(alive))[0, 0], h[0, 1])          # the finished agent is not heard
test_eq(mean_of_others(h, comm_mask(alive))[0, 2], torch.zeros(4))      # nor hears
far = comm_mask(torch.ones(1, 3, dtype=torch.bool), torch.tensor([[[0, 0], [2, 1], [5, 5]]]), comm_range=3)
test_eq(far[0], torch.tensor([[True, True, False], [True, True, False], [False, False, True]]))
test_eq(comm_mask(torch.ones(2, 3, dtype=torch.bool), drop_prob=1.0).any(), False)

### Messages over comm-core networks

CommNet's messages are hidden states inside one forward pass, and gradients flow through them, so
they are not sent through `CommunicationWrapper` (whose payloads are numpy arrays exchanged between
env steps). `NetworkLinks` uses comm-core for what it models, whether a message arrives: at each of
the `K - 1` rounds of an env step, every alive agent broadcasts a message of `hidden_dim` numbers
(its size for comm-core's channel and metrics) over its env's `AgentNetwork`, and the messages that
reach agent `i` from agent `k` set `mask[i, k]` for that round. The averaging stays in torch, over
the messages that arrived. The networks get the agents' cells (`set_positions`, for channels that
depend on where they are) and, with `radius`, a topology linking the agents at most `radius`
apart (`within_range`). Channels that alter the payloads (e.g. noise) only count here through the
messages they lose.

In [ ]:
#| export
class NetworkLinks:
    "Which CommNet messages arrive, through comm-core networks (`AgentNetwork`, one per env, with its own copy of `network_kwargs`)."
    def __init__(
        self,
        num_agents: int, # Agents per env
        radius: float | None = None, # Link the agents at most this far apart, at every round (`within_range`); None: the `topology`
        ord: float = np.inf, # Norm of `radius` (inf: Chebyshev, squares on the grid)
        broadcast: bool = True, # Each agent broadcasts its message once (one emission), instead of a unicast per neighbour
        **network_kwargs, # `AgentNetwork`'s arguments: topology, channel, blocks, noise_power, backend, protocol, bits_per_element
    ):
        self.num_agents, self.broadcast, self.network_kwargs = num_agents, broadcast, network_kwargs
        self._topology = within_range(radius, positions=lambda position: position, ord=ord) if radius is not None else None
        self.networks: list[AgentNetwork] = []

    def _network(self) -> AgentNetwork:
        return AgentNetwork(self.num_agents, **copy.deepcopy(self.network_kwargs))

    def reset(self, envs: Sequence[int], seeds: Sequence[int | None] | None = None):
        "A new episode in `envs` (indices of the pool); `seeds` reseed their channels."
        for k, e in enumerate(envs):
            while len(self.networks) <= e:
                self.networks.append(self._network())
            self.networks[e].reset(None if seeds is None else seeds[k])

    def masks(
        self,
        position: np.ndarray, # (B, N, 2) the agents' cells
        alive: np.ndarray, # (B, N) agents that send (and receive)
        rounds: int, # Rounds of messages
        message_dim: int, # Numbers per message
    ) -> torch.Tensor:
        "(B, rounds, N, N) bool: `mask[b, r, i, k]` when agent `k`'s message of round `r` reached agent `i`."
        B, N = alive.shape
        if len(self.networks) < B:
            self.reset(range(len(self.networks), B))
        mask = torch.zeros(B, rounds, N, N, dtype=torch.bool)
        message = np.zeros(message_dim, np.float32)
        for b in range(B):
            network = self.networks[b]
            if self._topology is not None:
                network.set_topology(self._topology(np.asarray(position[b], np.float64)))
            network.set_positions(np.asarray(position[b], np.float64))
            for r in range(rounds):
                received = network.exchange({a: message if alive[b, a] else None for a in range(N)},
                                            broadcast=self.broadcast)
                for i, senders in received.items():
                    mask[b, r, i, list(senders)] = True
        return mask

    @property
    def metrics(self) -> list:
        "comm-core's metrics of the current episode of each env."
        return [network.metrics for network in self.networks]

In [ ]:
#| hide
import comm_core as cc
alive = np.array([[True, True, True], [True, False, True]])
position = np.array([[[0, 0], [1, 1], [5, 5]], [[0, 0], [1, 1], [2, 2]]])
links = NetworkLinks(3)
m = links.masks(position, alive, rounds=2, message_dim=4)
test_eq(m.shape, (2, 2, 3, 3))
test_eq(m[0, 0], ~torch.eye(3, dtype=torch.bool))                       # a perfect full mesh: everyone hears everyone
test_eq(m[1, :, :, 1].any(), False)                                     # the finished agent sends nothing
test_eq((m & comm_mask(torch.as_tensor(alive)).unsqueeze(1))[1, :, 1].any(), False)   # and CommNet's mask drops what reaches it
test_eq(NetworkLinks(3, radius=2).masks(position, alive, 1, 4)[0, 0],
        torch.tensor([[False, True, False], [True, False, False], [False, False, False]]))
lossy = NetworkLinks(3, channel=cc.ErasureChannel(loss_rate=1.0))
test_eq(lossy.masks(position, alive, 1, 4).any(), False)
test_eq(len(lossy.metrics), 2)

### The network

`comm_steps` is the paper's `K` (modules `f^0 ... f^{K-1}`). The mask is one for all rounds
(B, N, N), or one per round (B, K - 1, N, N), e.g. from `NetworkLinks`. With `share_weights` the modules
share `H`, `C` and `S` (the paper's tied variant, which also allows another `K` at test time),
otherwise each step has its own. The policy head gives each agent's action logits and the baseline
head its `b(s, theta)`.

In [ ]:
#| export
class CommNet(nn.Module):
    "CommNet: an encoder, `comm_steps` communication steps and per-agent policy and baseline heads, shared by the agents."
    def __init__(
        self,
        obs_dim: int, # Features per agent (see `commnet_features`)
        num_actions: int = 4, # Actions per agent (MultiGrid's navigation: left, right, forward, done)
        hidden_dim: int = 128, # Width of the hidden states (the messages)
        comm_steps: int = 2, # K: communication steps (agents exchange messages K - 1 times)
        skip: bool = True, # The paper's skip connection from h^0 into every step
        share_weights: bool = False, # One module for all steps (the paper's tied variant)
        obs_keys: Sequence[str] = ('image', 'direction'), # Each agent's observations (info keys)
        goal: bool = True, # Observe the offset to the goal cell
        grid_size: int = 15, # Side of the grid (scales positions)
        comm_range: int | None = None, # Agents hear the agents within this many cells (None: all)
        drop_prob: float = 0.0, # Probability of a lost message (each pair and step)
    ):
        super().__init__()
        if comm_steps < 1:
            raise ValueError(f"comm_steps must be >= 1, got {comm_steps}")
        self.config = dict(obs_dim=obs_dim, num_actions=num_actions, hidden_dim=hidden_dim, comm_steps=comm_steps,
                           skip=skip, share_weights=share_weights, obs_keys=list(obs_keys), goal=goal,
                           grid_size=grid_size, comm_range=comm_range, drop_prob=drop_prob)
        self.comm_steps, self.share_weights = comm_steps, share_weights
        self.obs_keys, self.goal, self.grid_size = tuple(obs_keys), goal, grid_size
        self.comm_range, self.drop_prob = comm_range, drop_prob
        modules = 1 if share_weights else comm_steps
        self.encoder = nn.Linear(obs_dim, hidden_dim)
        self.H = nn.ModuleList(nn.Linear(hidden_dim, hidden_dim) for _ in range(modules))
        # c^0 = 0: the first step has no message, so step j > 0 uses C[j - 1] (or the shared C)
        self.C = nn.ModuleList(nn.Linear(hidden_dim, hidden_dim, bias=False)
                               for _ in range(0 if comm_steps == 1 else 1 if share_weights else comm_steps - 1))
        self.S = nn.ModuleList(nn.Linear(hidden_dim, hidden_dim, bias=False) for _ in range(modules)) if skip else None
        self.policy_head = nn.Linear(hidden_dim, num_actions)
        self.baseline_head = nn.Linear(hidden_dim, 1)
        self.temperature = 1.0      # acting: 0 picks the most likely action, otherwise samples (see `get_action`)
        self.checkpoint: dict = {}  # what `save` stored besides the model (e.g. the env)
        self.links: NetworkLinks | None = None   # messages over comm-core networks (not saved with the model)

    def forward(
        self,
        obs: torch.Tensor, # (B, N, obs_dim)
        mask: torch.Tensor | None = None, # (B, N, N) or per round (B, K - 1, N, N): who hears whom; None: everyone
    ) -> tuple[torch.Tensor, torch.Tensor]:
        "Action logits (B, N, num_actions) and baselines (B, N)."
        h0 = torch.tanh(self.encoder(obs))
        h = h0
        for j in range(self.comm_steps):
            m = 0 if self.share_weights else j
            out = self.H[m](h)
            if j > 0:
                heard = mask[:, j - 1] if mask is not None and mask.dim() == 4 else mask
                out = out + self.C[0 if self.share_weights else j - 1](mean_of_others(h, heard))
            if self.S is not None:
                out = out + self.S[m](h0)
            h = torch.tanh(out)
        return self.policy_head(h), self.baseline_head(h).squeeze(-1)

In [ ]:
#| hide
torch.manual_seed(0)
obs = torch.randn(2, 3, 5)
other = obs.clone(); other[:, 1] += 1.0
for K, depends in ((1, False), (2, True), (3, True)):
    net = CommNet(5, comm_steps=K, hidden_dim=8)
    changed = not torch.allclose(net(obs)[0][:, 0], net(other)[0][:, 0])
    test_eq(changed, depends)          # agent 0 hears agent 1 only if there is a message (K > 1)
net = CommNet(5, comm_steps=3, hidden_dim=8, share_weights=True)
test_eq(len(net.H), 1); test_eq(len(net.C), 1)
deaf = comm_mask(torch.ones(2, 3, dtype=torch.bool), torch.tensor([[[0, 0], [9, 9], [0, 1]]] * 2), comm_range=2)
test_close(net(obs, deaf)[0][:, 0], net(other, deaf)[0][:, 0])   # agent 1 is out of range of agent 0
logits, baseline = net(obs)
test_eq(logits.shape, (2, 3, 4)); test_eq(baseline.shape, (2, 3))

### Acting

As the model of a `FeedForwardPolicy`, `get_action` reads every agent's observations from the info
dict of a pool of envs, so the agents of each env communicate within one forward pass. Agents that
already reached their goal (`terminated`) are left out of the communication. With `links`, the
networks of the envs that reset (`_needs_flush`) start a new episode, seeded with the env's `seed`.

In [ ]:
#| export
@patch
def _info_keys(self: CommNet) -> list[str]:
    keys = [*self.obs_keys, 'terminated']
    if self.goal:
        keys += ['position', 'goal_position']
    if self.comm_range is not None or self.links is not None:
        keys += ['position']
    return list(dict.fromkeys(keys))


@patch
def observe(self: CommNet, columns: dict[str, np.ndarray]) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Features (B, N, obs_dim) and the communication mask of columns of one step (B, N, ...): (B, N, N),
    or per round (B, K - 1, N, N) with `links` (which also sends this step's messages).
    """
    obs = torch.from_numpy(commnet_features(columns, self.obs_keys, self.grid_size, self.goal))
    position = torch.as_tensor(np.asarray(columns['position'])) if self.comm_range is not None else None
    alive = ~np.asarray(columns['terminated'], bool)
    mask = comm_mask(torch.as_tensor(alive), position, self.comm_range, self.drop_prob)
    if self.links is not None and self.comm_steps > 1:
        delivered = self.links.masks(np.asarray(columns['position']), alive, self.comm_steps - 1, self.config['hidden_dim'])
        mask = mask.unsqueeze(1) & delivered
    return obs, mask


@patch
@torch.no_grad()
def get_action(self: CommNet, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) for the info dict of a pool of envs (`(num_envs, 1, num_agents, ...)` columns)."
    columns = {k: torch.as_tensor(info[k])[:, -1].cpu().numpy() for k in self._info_keys()}
    if self.links is not None:
        flush = info.get('_needs_flush')
        envs = range(len(columns['terminated'])) if flush is None else np.flatnonzero(torch.as_tensor(flush).cpu().numpy())
        seeds = torch.as_tensor(info['seed'])[:, -1].cpu().numpy() if 'seed' in info else None
        self.links.reset(envs, None if seeds is None else [int(seeds[e]) for e in envs])
    obs, mask = self.observe(columns)
    device = next(self.parameters()).device
    logits, _ = self(obs.to(device), mask.to(device))
    if self.temperature > 0:
        return torch.distributions.Categorical(logits=logits / self.temperature).sample()
    return logits.argmax(-1)

### Training online (the paper)

`train_commnet` follows the paper's Appendix A: every update plays one episode in each env of a
`World` (sampling the actions), then follows the gradient of
`sum_t [log pi(a_t | s_t) (R_t - b(s_t)) - alpha (R_t - b(s_t))^2]`, with `R_t` the undiscounted sum
of the rewards from `t` to the end of the episode (no bootstrapping) and `alpha = 0.03`, over the
steps where the agents act. Each agent's return is its own rewards, or with `team_return` the sum
over the agents (every agent then credits the team's success). The envs get new seeds at every
update, so every episode is a new layout. With `links`, the messages of the rollouts go over the
networks, and the update uses the masks of what arrived.

In [ ]:
#| export
def rollout(model: CommNet, world, seed: int) -> tuple[dict[str, torch.Tensor], dict[str, np.ndarray]]:
    """
    One episode in each env of `world` with actions sampled from `model`: its steps
    (`obs`, `mask`, `alive`, `action`, `reward`, each (T, E, N, ...)) and per env `success`
    (every agent reached its goal), `return` and `length`.
    """
    E = world.num_envs
    world.reset(seed=[seed + i for i in range(E)])
    if model.links is not None:
        model.links.reset(range(E), [seed + i for i in range(E)])
    running = np.ones(E, bool)
    reached = np.zeros((E, world.num_agents), bool)
    length = np.zeros(E, np.int64)
    steps: dict[str, list] = {k: [] for k in ('obs', 'mask', 'alive', 'action', 'reward')}
    device = next(model.parameters()).device
    while running.any():
        columns = {k: np.asarray(world.infos[k])[:, -1].copy() for k in model._info_keys()}
        columns['terminated'] = columns['terminated'] | ~running[:, None]   # finished envs: nobody acts
        obs, mask = model.observe(columns)
        with torch.no_grad():
            logits, _ = model(obs.to(device), mask.to(device))
        action = torch.distributions.Categorical(logits=logits).sample().cpu()
        _, rewards, terms, truncs, world.infos = world.envs.step(action.numpy(), mask=running)
        alive = torch.from_numpy(~columns['terminated'])
        for k, v in (('obs', obs), ('mask', mask), ('alive', alive), ('action', action),
                     ('reward', torch.from_numpy(rewards) * alive)):
            steps[k].append(v)
        reached |= terms & running[:, None]
        length += running
        running &= ~episode_done(terms, truncs)
    batch = {k: torch.stack(v) for k, v in steps.items()}
    return batch, {'success': reached.all(1), 'return': batch['reward'].sum((0, 2)).numpy(), 'length': length}


def reinforce_losses(
    model: CommNet,
    batch: dict[str, torch.Tensor], # Steps of `rollout`, (T, E, N, ...)
    baseline_coef: float = 0.03, # The paper's alpha
    entropy_coef: float = 0.0, # Entropy bonus (not in the paper)
    team_return: bool = False, # Every agent's return is the sum over the agents
) -> dict[str, torch.Tensor]:
    "`loss` (to minimize), `policy_loss`, `baseline_loss` and `entropy` of a batch of episodes."
    T, E, N = batch['action'].shape
    logits, baseline = model(batch['obs'].flatten(0, 1), batch['mask'].flatten(0, 1))
    logits, baseline = logits.view(T, E, N, -1), baseline.view(T, E, N)
    reward = batch['reward'].float()
    if team_return:
        reward = reward.sum(-1, keepdim=True).expand_as(reward)
    returns = reward.flip(0).cumsum(0).flip(0)                    # R_t: rewards from t to the end
    weight = batch['alive'].float()
    n = weight.sum().clamp(min=1)
    dist = torch.distributions.Categorical(logits=logits)
    advantage = returns - baseline
    policy_loss = -(dist.log_prob(batch['action']) * advantage.detach() * weight).sum() / n
    baseline_loss = (advantage.pow(2) * weight).sum() / n
    entropy = (dist.entropy() * weight).sum() / n
    return {'loss': policy_loss + baseline_coef * baseline_loss - entropy_coef * entropy,
            'policy_loss': policy_loss.detach(), 'baseline_loss': baseline_loss.detach(), 'entropy': entropy.detach()}


def train_commnet(
    model: CommNet,
    world, # A `World` (goal-conditioned for `goal`): every update plays one episode per env
    episodes: int = 20_000, # Episodes played in total (updates: episodes / world.num_envs)
    lr: float = 1e-3,
    baseline_coef: float = 0.03, # The paper's alpha
    entropy_coef: float = 0.0,
    team_return: bool = False,
    max_grad_norm: float | None = None, # Clip the gradient norm (not in the paper)
    seed: int = 0, # Seeds of the episodes: seed, seed + 1, ...
    log_every: int = 20, # Updates between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the episodes played and the means since the last call
) -> CommNet:
    "Train `model` online with REINFORCE and a learned baseline (the paper's Appendix A); returns it in eval mode."
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    E = world.num_envs
    totals: dict[str, float] = {}
    count = 0
    for update in range(1, episodes // E + 1):
        batch, stats = rollout(model, world, seed + (update - 1) * E)
        batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}
        losses = reinforce_losses(model, batch, baseline_coef, entropy_coef, team_return)
        optimizer.zero_grad(set_to_none=True)
        losses['loss'].backward()
        if max_grad_norm is not None:
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        row = {**{k: float(v) for k, v in losses.items()}, 'success_rate': float(stats['success'].mean()),
               'return': float(stats['return'].mean()), 'length': float(stats['length'].mean())}
        for k, v in row.items():
            totals[k] = totals.get(k, 0.0) + v
        count += 1
        if log is not None and (update % log_every == 0 or update == episodes // E):
            log({'episodes': update * E, **{k: v / count for k, v in totals.items()}})
            totals, count = {}, 0
    return model.eval()

### Training from a dataset

`CommNetDataset` makes one sample per step of every episode of a dataset (collected by any
policy, e.g. `MixedPolicy`): all the agents' observations together (they communicate), with the
shortest-path expert's actions towards their goals as labels (recomputed from the recorded grid,
see `shortest_path_actions`), ignored for agents that already reached it.
`train_commnet_supervised` minimizes their cross-entropy, with the communication mask of the
model (its range and lost messages) drawn for every sample; the samples are not in envs, so `links`
are not used (a model can still act with them, e.g. to evaluate it under a channel it was not
trained with).

In [ ]:
#| export
class CommNetDataset(torch.utils.data.Dataset):
    "Joint samples (all the agents at one step) of a MultiGrid dataset, labelled with the expert's actions."
    def __init__(self, dataset, model: CommNet):
        keys = list(dict.fromkeys([*model._info_keys(), 'position', 'direction', 'goal_position', 'state']))
        cols = {k: dataset.get_col_data(k) for k in keys}
        self.obs = torch.from_numpy(commnet_features(cols, model.obs_keys, model.grid_size, model.goal))
        self.alive = torch.from_numpy(~np.asarray(cols['terminated'], bool))
        self.position = torch.from_numpy(np.asarray(cols['position'], np.int64))
        labels = np.full(self.alive.shape, IGNORE_INDEX, np.int64)
        for start, length in zip(np.asarray(dataset.offsets), np.asarray(dataset.lengths)):
            for t in range(start, start + length):
                walkable = walkable_cells(cols['state'][t])
                for a in np.flatnonzero(self.alive[t].numpy()):
                    actions = shortest_path_actions(walkable, cols['position'][t, a], cols['direction'][t, a],
                                                    cols['goal_position'][t, a])
                    if actions is not None and len(actions):
                        labels[t, a] = actions[0]
        self.labels = torch.from_numpy(labels)
        self.rows = torch.from_numpy(np.flatnonzero((labels != IGNORE_INDEX).any(1)))

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict[str, torch.Tensor]:
        r = self.rows[i]
        return {'obs': self.obs[r], 'alive': self.alive[r], 'position': self.position[r], 'actions': self.labels[r]}


def train_commnet_supervised(
    model: CommNet,
    data: CommNetDataset,
    steps: int = 5_000, # Gradient steps
    batch_size: int = 64,
    lr: float = 1e-3,
    device: str | torch.device | None = None, # None: cuda if available
    log_every: int = 200, # Steps between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the step and the means since the last call
) -> CommNet:
    "Train `model` to imitate the labels of `data` (cross-entropy); returns it in eval mode."
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model.to(device).train()
    sampler = torch.utils.data.RandomSampler(data, replacement=True, num_samples=steps * batch_size)
    loader = torch.utils.data.DataLoader(data, batch_size, sampler=sampler, drop_last=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    totals: dict[str, float] = {}
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        mask = comm_mask(batch['alive'], batch['position'], model.comm_range, model.drop_prob)
        logits, _ = model(batch['obs'], mask)
        labels = batch['actions']
        loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=IGNORE_INDEX)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        valid = labels != IGNORE_INDEX
        row = {'loss': float(loss), 'accuracy': float((logits.argmax(-1) == labels)[valid].float().mean())}
        for k, v in row.items():
            totals[k] = totals.get(k, 0.0) + v
        if log is not None and (step % log_every == 0 or step == steps):
            n = step % log_every or log_every
            log({'step': step, **{k: v / n for k, v in totals.items()}})
            totals = {}
    return model.eval()

### Saving and loading

In [ ]:
#| export
@patch
def save(self: CommNet, path: str | Path, **extra: Any) -> Path:
    "Save the config, the weights and `extra` (e.g. `env_name`, `env_kwargs`) to `path` (e.g. `commnet.pt`)."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'config': self.config, 'state_dict': self.state_dict(), **extra}, path)
    return path


@patch(cls_method=True)
def load(cls: CommNet, path: str | Path, map_location: Any = 'cpu') -> CommNet:
    "A `CommNet` saved with `save`; what it stored besides the model is in `checkpoint`."
    checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    model = cls(**checkpoint.pop('config'))
    model.load_state_dict(checkpoint.pop('state_dict'))
    model.checkpoint = checkpoint
    return model.eval()

### Tests

Online: a few updates on a small FindGoal; from a dataset: `MixedPolicy` episodes, labelled by the
expert; both models act in a `World`.

In [ ]:
#| hide
import tempfile
from stable_marl import World
from stable_marl.data import HDF5Dataset
from stable_marl.envs.multigrid import MixedPolicy
from stable_marl.policy import FeedForwardPolicy

kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=15)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=3, goal_conditioned=True, **kw)
world.reset(seed=[0, 1, 2])
dim = commnet_features({k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}, grid_size=7).shape[-1]

torch.manual_seed(0)
model = CommNet(dim, hidden_dim=32, grid_size=7, comm_range=3)
batch, stats = rollout(model, world, seed=0)
T = batch['action'].shape[0]
test_eq(batch['obs'].shape, (T, 3, 2, dim)); test_eq(batch['mask'].shape, (T, 3, 2, 2))
test_eq(int(stats['length'].max()), T)
assert (batch['reward'][~batch['alive']] == 0).all()
logs = []
train_commnet(model, world, episodes=12, log_every=2, log=logs.append)
test_eq([r['episodes'] for r in logs], [6, 12])
assert all(np.isfinite(r['loss']) for r in logs)

world.set_policy(FeedForwardPolicy(model))
test_eq(len(world.evaluate(episodes=3, seed=100)['episode_successes']), 3)

# over comm-core networks (no range here): a lossless channel delivers every message, a dead one none
model.comm_range = None
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=0.0))
batch, _ = rollout(model, world, seed=5)
test_eq(batch['mask'].shape[2:], (1, 2, 2))          # (T, E, rounds, N, N)
test_eq((batch['mask'][:, :, 0, 0, 1] == batch['alive'].all(-1)).all(), True)
m = model.links.metrics[0]
assert m.transmissions > 0; test_eq(m.bits_sent, m.transmissions * 32 * 32)    # hidden_dim float32 numbers each
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=1.0))
batch, _ = rollout(model, world, seed=5)
test_eq(batch['mask'].any(), False)
train_commnet(model, world, episodes=3, log_every=1)
test_eq(len(world.evaluate(episodes=3, seed=100)['episode_successes']), 3)
model.links, model.comm_range = None, 3

path = model.save(f'{tempfile.mkdtemp()}/commnet.pt', env_name='MultiGrid-FindGoal-15x15-v0', env_kwargs=kw)
again = CommNet.load(path)
test_eq(again.checkpoint['env_kwargs'], kw); test_eq(again.comm_range, 3)
obs = torch.randn(2, 2, dim)
test_close(again(obs)[0], model(obs)[0])

In [ ]:
#| hide
tmp = tempfile.mkdtemp()
collector = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
collector.set_policy(MixedPolicy(seed=0))
collector.collect(f'{tmp}/mixed.h5', episodes=8, seed=0, progress=False)
ds = HDF5Dataset(path=f'{tmp}/mixed.h5')

torch.manual_seed(0)
model = CommNet(dim, hidden_dim=32, grid_size=7)
data = CommNetDataset(ds, model)
sample = data[0]
test_eq(sample['obs'].shape, (2, dim)); test_eq(sample['actions'].shape, (2,))
test_eq(((data.labels != IGNORE_INDEX) <= data.alive).all(), True)      # labels only for agents still acting
logs = []
train_commnet_supervised(model, data, steps=60, batch_size=16, lr=3e-3, log_every=30, log=logs.append, device='cpu')
assert logs[-1]['loss'] < logs[0]['loss']
model.temperature = 0.0
world.set_policy(FeedForwardPolicy(model))
test_eq(len(world.evaluate(episodes=3, seed=100)['episode_successes']), 3)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()